# 04 · Neighbourhood queries on the sphere

`get_neighbours(v)` returns the immediate neighbours of a vertex and `get_neighbours_in_distance(v, d)`
returns *d* "rings" around it. Seam copies are handled for you, so a ring continues seamlessly across
the edges of the net.

| centre | ring sizes |
|---|---|
| interior or seam vertex | 6, 12, 18, 24, … |
| icosahedron corner | 5, 10, 15, 20, … |

**Important:** the search marks vertices as visited. Call `dome.unmark_vertices()` before every new query
(the helper `dome_utils.neighbour_rings` does this for you).

Script version: [`examples/04_neighbours.py`](../04_neighbours.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from dome_utils import neighbour_rings, unique_mesh
from IPython.display import display
from matplotlib.colors import to_rgb
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from nb_setup import LiveFigure

from mt.geodesicdome.grid.geodesicdome import GeodesicDome

## The basic API

In [ ]:
FREQ = 10
dome = GeodesicDome(FREQ)
points, faces, index_map = unique_mesh(dome)       # also assigns vertex.id

v = dome.get_vertex_at(15, 17)
dome.unmark_vertices()
first = dome.get_neighbours(v, False)
print(f'vertex ({v.x},{v.y}) has {len(first)} neighbours at grid positions', [(n.x, n.y) for n in first])

dome.unmark_vertices()
rings = dome.get_neighbours_in_distance(v, 3)
print('ring sizes:', [len(r) for r in rings])

## Drawing the rings

Each unique point gets the number of the ring it belongs to; faces take the colour of their outermost
ring. Only the hemisphere facing the camera is drawn, with the camera looking at the centre vertex.

In [ ]:
COLOURS = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60', '#2980b9', '#8e44ad', '#16a085', '#d35400', '#7f8c8d']


def draw_rings(ax, dome, points, faces, index_map, centre, n_rings, label=''):
    rings = neighbour_rings(dome, centre, n_rings)
    sizes = [len(r) for r in rings]

    rank = np.full(len(points), n_rings + 1)              # n_rings + 1 = outside the neighbourhood
    for r, ring in enumerate(rings, start=1):
        rank[[index_map[u.id] for u in ring]] = r
    rank[index_map[centre.id]] = 0

    palette = np.array([to_rgb(COLOURS[i % len(COLOURS)]) for i in range(n_rings + 1)] + [(0.85, 0.85, 0.85)])
    face_rank = rank[faces].max(axis=1)
    face_colour = 0.5 + 0.5 * palette[face_rank]
    face_colour[face_rank > n_rings] = (0.93, 0.94, 0.96)

    view = centre.coord / np.linalg.norm(centre.coord)
    tri = points[faces]
    visible = tri.mean(axis=1) @ view > 0.05
    ax.add_collection3d(Poly3DCollection(tri[visible], facecolors=face_colour[visible],
                                         edgecolors='#9aa5b1', linewidths=0.3))
    front = points @ view > 0.05
    ax.scatter(*points[front].T, c=palette[rank][front], s=12, depthshade=False)
    ax.set_xlim(-1, 1), ax.set_ylim(-1, 1), ax.set_zlim(-1, 1)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=np.degrees(np.arcsin(view[2])), azim=np.degrees(np.arctan2(view[1], view[0])))
    ax.set_axis_off()
    ax.set_title(f'{label}\nring sizes {sizes}')
    return sizes


def ring_legend(fig, n_rings):
    handles = [plt.Line2D([], [], marker='o', ls='', color=COLOURS[i % len(COLOURS)],
                          label='centre' if i == 0 else f'ring {i}') for i in range(n_rings + 1)]
    fig.legend(handles=handles, loc='lower center', ncol=min(n_rings + 1, 9))

## An interior vertex, a seam vertex and an icosahedron corner

In [ ]:
RINGS = 4
cases = [('interior vertex', dome.get_vertex_at(15, 17)),
         ('seam vertex (stored twice)', dome.get_vertex_at(0, FREQ // 2)),   # middle of a net edge
         ('icosahedron corner', dome.get_vertex_at(0, 0))]

fig = plt.figure(figsize=(15, 5.2))
for n, (label, centre) in enumerate(cases, start=1):
    sizes = draw_rings(fig.add_subplot(1, 3, n, projection='3d'), dome, points, faces, index_map,
                       centre, RINGS, label)
    print(f'{label:28s} grid ({centre.x:2d},{centre.y:2d})  ring sizes {sizes}')
ring_legend(fig, RINGS)
fig.suptitle(f'get_neighbours_in_distance(v, {RINGS}) on GeodesicDome(frequency={FREQ})', fontsize=14)
fig.subplots_adjust(left=0, right=1, bottom=0.1, top=0.83, wspace=0)
plt.show()

### Try it

Choose the frequency, the number of rings and the centre vertex (by its index in
`dome.get_all_vertices()`, or with the shortcut buttons). Watch the ring sizes drop from 6, 12, 18, …
to 5, 10, 15, … when the ring touches an icosahedron corner.

In [ ]:
freq = widgets.IntSlider(value=10, min=2, max=20, description='frequency')
n_rings = widgets.IntSlider(value=4, min=1, max=8, description='rings')
index = widgets.IntSlider(value=0, min=0, max=1, description='centre')
preset = widgets.ToggleButtons(options=['interior', 'seam', 'corner', 'slider'], value='interior',
                               description='centre is')
cache = {}


def mesh(f):
    if f not in cache:
        d = GeodesicDome(f)
        cache[f] = (d, *unique_mesh(d))
    return cache[f]


def sync_index(*_):
    index.max = len(mesh(freq.value)[0].get_all_vertices()) - 1


freq.observe(sync_index, names='value')
sync_index()
live = LiveFigure(figsize=(7, 7))


def draw(freq, n_rings, index, preset):
    d, pts, fcs, imap = mesh(freq)
    vs = d.get_all_vertices()
    if preset == 'interior':                       # a vertex stored once, half-way through the list
        centre = next(u for u in vs[len(vs) // 2:] + vs if not u.same_vertices)
    elif preset == 'seam':                         # middle of a net edge
        centre = d.get_vertex_at(0, max(1, freq // 2))
    elif preset == 'corner':
        centre = d.get_vertex_at(0, 0)
    else:
        centre = d.get_all_vertices()[index]
    fig = live.clear()
    sizes = draw_rings(fig.add_subplot(projection='3d'), d, pts, fcs, imap, centre, n_rings,
                       f'GeodesicDome({freq}), centre at grid ({centre.x}, {centre.y})')
    ring_legend(fig, n_rings)
    fig.subplots_adjust(left=0, right=1, bottom=0.08, top=0.9)
    live.refresh()
    print('ring sizes', sizes)


out = widgets.interactive_output(draw, dict(freq=freq, n_rings=n_rings, index=index, preset=preset))
display(widgets.VBox([widgets.HBox([freq, n_rings]), widgets.HBox([preset, index])]), out)
live.show()